# 03 · Hand-calculated exact checks + exclusions/scaling (theme C)

Everything here is checked against **closed-form expectations**, with atoms at
hand-chosen coordinates and preset charges. Three families:

1. **Virtual-site geometry** vs the SMIRNOFF definitions (distance from the
parent atom; out-of-plane angle for `DivalentLonePair` → interchange #1438;
in-plane angle for `MonovalentLonePair` → interchange #939).
2. **Charge-increment bookkeeping** — parent charge is reduced by the increment,
the vsite carries it, total charge is exactly conserved.
3. **Nonbonded exclusions & 1-4 scaling** for atom–atom, atom–vsite and
vsite–vsite pairs, including **non-default 1-4 scaling** (interchange #1436)
and the `combine_nonbonded_forces=False` layout (interchange #1498), each
compared to hand-computed values.

A short analytic Coulomb+LJ energy check closes the loop.

In [1]:
import warnings

import numpy as np
import openmm
import openmm.unit as u
from openff.toolkit import ForceField, Molecule
from openff.units import unit as offunit

import _helpers as H
import pathways as P

warnings.filterwarnings("ignore")
np.set_printoptions(precision=6, suppress=True)
RESULTS = []
ANG = offunit.angstrom


def build_ic(smiles, coords_A, vsite_xml, vdw, libq, **off_kwargs):
    """Interchange OpenMM system for a molecule at explicit coordinates (angstrom)."""
    mol = Molecule.from_mapped_smiles(smiles)
    mol.add_conformer(np.asarray(coords_A) * ANG)
    ff = ForceField(P.build_offxml(vdw, libq, [vsite_xml], **{
        k: v for k, v in off_kwargs.items() if k in ("elec_scale14", "vdw_scale14")}))
    system = ff.create_interchange(mol.to_topology()).to_openmm_system(
        combine_nonbonded_forces=off_kwargs.get("combine_nonbonded_forces", True))
    positions = H.resolved_vsite_positions(system, mol.conformers[0].to_openmm())
    return mol, system, positions


def vsite_indices(system):
    return [i for i in range(system.getNumParticles()) if system.isVirtualSite(i)]


VDW0 = [("[#6:1]", 3.0, 0.0), ("[#1:1]", 2.0, 0.0),
        ("[#7:1]", 3.0, 0.0), ("[#8:1]", 3.0, 0.0), ("[#17:1]", 3.0, 0.0)]
LIBQ0 = [("[#6:1]", 0.0), ("[#1:1]", 0.0), ("[#7:1]", 0.0),
         ("[#8:1]", 0.0), ("[#17:1]", 0.0)]

## 1 · Geometry — closed-form placement

### BondCharge: collinear
Cl at origin, C on the −x axis. With `distance=d>0` the site lies a distance `d`
beyond Cl (the atom tagged `:1`), i.e. exactly at `(d, 0, 0)`.

In [2]:
def check_bondcharge():
    d = 0.5
    coords = [[0, 0, 0], [-1.7, 0, 0], [-2.1, 0.9, 0], [-2.1, -0.5, 0.8], [-2.1, -0.5, -0.8]]
    vs = (f'<VirtualSite smirks="[#17:1]-[#6:2]" type="BondCharge" name="EP" match="all_permutations"'
          f' distance="{d}*angstrom" charge_increment1="0.0*elementary_charge"'
          f' charge_increment2="0.0*elementary_charge" sigma="0*angstrom" epsilon="0*kilocalorie/mole"/>')
    mol, sys, pos = build_ic("[Cl:1][C:2]([H:3])([H:4])[H:5]", coords, vs, VDW0, LIBQ0)
    S = pos[vsite_indices(sys)[0]] * 10.0  # nm -> A
    expected = np.array([d, 0, 0])
    err = np.linalg.norm(S - expected)
    print(f"BondCharge: placed {S}, expected {expected}, |err|={err*1:.2e} A")
    assert err < 1e-5, f"BondCharge placement off by {err} A"
    RESULTS.append({"notebook": "03", "check": "BondCharge geometry", "error_A": err, "status": "PASS"})


check_bondcharge()

BondCharge: placed [0.5 0.  0. ], expected [0.5 0.  0. ], |err|=0.00e+00 A


### TrivalentLonePair: on the C3 axis
N at origin, three H in the plane `z=-0.4`. The site lies distance `d` along the
perpendicular to that plane through N, on the side opposite the substituents:
exactly `(0, 0, +d)`.

In [3]:
def check_trivalent():
    d = 0.3
    r, zc = 1.0, -0.4
    hs = [[r * np.cos(a), r * np.sin(a), zc] for a in (0, 2 * np.pi / 3, 4 * np.pi / 3)]
    coords = [[0, 0, 0]] + hs
    vs = (f'<VirtualSite smirks="[#1:2][#7:1]([#1:3])[#1:4]" type="TrivalentLonePair" name="LP"'
          f' match="once" distance="{d}*angstrom" charge_increment1="0.0*elementary_charge"'
          f' charge_increment2="0.0*elementary_charge" charge_increment3="0.0*elementary_charge"'
          f' charge_increment4="0.0*elementary_charge" sigma="0*angstrom" epsilon="0*kilocalorie/mole"/>')
    mol, sys, pos = build_ic("[N:1]([H:2])([H:3])[H:4]", coords, vs, VDW0, LIBQ0)
    S = pos[vsite_indices(sys)[0]] * 10.0
    dist = np.linalg.norm(S)
    on_axis = np.linalg.norm(S[:2])
    print(f"TrivalentLonePair: S={S}, |S-N|={dist:.6f} A (expect {d}), off-axis={on_axis:.2e}, z-sign={np.sign(S[2])}")
    assert abs(dist - d) < 1e-5 and on_axis < 1e-5 and S[2] > 0
    RESULTS.append({"notebook": "03", "check": "TrivalentLonePair geometry",
                    "error_A": abs(dist - d), "status": "PASS"})


check_trivalent()

TrivalentLonePair: S=[ 0.  -0.   0.3], |S-N|=0.300000 A (expect 0.3), off-axis=1.30e-16, z-sign=1.0


### DivalentLonePair: distance + out-of-plane angle (interchange #1438)
O at origin; two neighbours symmetric about the x-axis in the xy-plane on the +x
side, so the angle bisector is +x and the site sits on the −x side (outside the
angle). With `match="all_permutations"` and `outOfPlaneAngle=θ` we get two sites
at distance `d` from O, lifted ±θ out of the xy-plane. We verify **both** the
distance and the out-of-plane angle — the quantities that were wrong in #1438.

In [4]:
def check_divalent(theta_deg):
    d = 0.6
    half = np.deg2rad(52.0)          # half of the H-O-H-like angle
    n2 = [np.cos(half), np.sin(half), 0]
    n3 = [np.cos(half), -np.sin(half), 0]
    coords = [[0, 0, 0], list(np.array(n2) * 1.0), list(np.array(n3) * 1.0)]
    vs = (f'<VirtualSite smirks="[#1:2][#8:1][#1:3]" type="DivalentLonePair" name="LP"'
          f' match="all_permutations" distance="{d}*angstrom" outOfPlaneAngle="{theta_deg}*degree"'
          f' charge_increment1="0.0*elementary_charge" charge_increment2="0.0*elementary_charge"'
          f' charge_increment3="0.0*elementary_charge" sigma="0*angstrom" epsilon="0*kilocalorie/mole"/>')
    mol, sys, pos = build_ic("[O:1]([H:2])[H:3]", coords, vs, VDW0, LIBQ0)
    sites = [pos[i] * 10.0 for i in vsite_indices(sys)]
    ok = True
    for S in sites:
        dist = np.linalg.norm(S)                       # distance from O
        oop = np.rad2deg(np.arcsin(abs(S[2]) / dist))  # angle above/below xy-plane
        inplane_x = S[0]                               # bisector is +x; site outside -> -x
        print(f"  theta={theta_deg}: |S-O|={dist:.6f} (expect {d}), out-of-plane={oop:.4f} deg "
              f"(expect {theta_deg}), x={inplane_x:+.4f} (expect <0)")
        ok = ok and abs(dist - d) < 1e-5 and abs(oop - theta_deg) < 1e-3 and inplane_x < 0
    assert ok, f"DivalentLonePair geometry wrong at theta={theta_deg}"
    RESULTS.append({"notebook": "03", "check": f"DivalentLonePair geometry (oop={theta_deg})",
                    "status": "PASS"})


print("DivalentLonePair (interchange #1438 regression):")
for theta in (0.0, 30.0, 54.75):
    check_divalent(theta)

DivalentLonePair (interchange #1438 regression):
  theta=0.0: |S-O|=0.600000 (expect 0.6), out-of-plane=0.0000 deg (expect 0.0), x=-0.6000 (expect <0)
  theta=0.0: |S-O|=0.600000 (expect 0.6), out-of-plane=0.0000 deg (expect 0.0), x=-0.6000 (expect <0)
  theta=30.0: |S-O|=0.600000 (expect 0.6), out-of-plane=30.0000 deg (expect 30.0), x=-0.5196 (expect <0)
  theta=30.0: |S-O|=0.600000 (expect 0.6), out-of-plane=30.0000 deg (expect 30.0), x=-0.5196 (expect <0)
  theta=54.75: |S-O|=0.600000 (expect 0.6), out-of-plane=54.7500 deg (expect 54.75), x=-0.3463 (expect <0)
  theta=54.75: |S-O|=0.600000 (expect 0.6), out-of-plane=54.7500 deg (expect 54.75), x=-0.3463 (expect <0)


### MonovalentLonePair: distance + in-plane angle (interchange #939)
O (`:1`) at origin, C (`:2`) on the −x axis, in the xy-plane. With
`outOfPlaneAngle=0` the site is in-plane at distance `d`, making the specified
`inPlaneAngle` with the O→C direction.

In [5]:
def check_monovalent(inplane_deg):
    d = 0.4
    coords = [[0, 0, 0], [-1.2, 0, 0], [-1.8, 0.9, 0], [-1.8, -0.9, 0]]
    vs = (f'<VirtualSite smirks="[#8:1]=[#6:2][#1:3]" type="MonovalentLonePair" name="LP"'
          f' match="all_permutations" distance="{d}*angstrom" inPlaneAngle="{inplane_deg}*degree"'
          f' outOfPlaneAngle="0*degree" charge_increment1="0.0*elementary_charge"'
          f' charge_increment2="0.0*elementary_charge" charge_increment3="0.0*elementary_charge"'
          f' sigma="0*angstrom" epsilon="0*kilocalorie/mole"/>')
    mol, sys, pos = build_ic("[O:1]=[C:2]([H:3])[H:4]", coords, vs, VDW0, LIBQ0)
    O = pos[0] * 10.0
    C = pos[1] * 10.0
    ok = True
    for i in vsite_indices(sys):
        S = pos[i] * 10.0
        dist = np.linalg.norm(S - O)
        v1 = (C - O) / np.linalg.norm(C - O)
        v2 = (S - O) / np.linalg.norm(S - O)
        ang = np.rad2deg(np.arccos(np.clip(v1 @ v2, -1, 1)))
        oop = abs(S[2])
        print(f"  inPlane={inplane_deg}: |S-O|={dist:.6f} (expect {d}), angle(S,O,C)={ang:.4f} "
              f"(expect {inplane_deg}), |z|={oop:.2e}")
        ok = ok and abs(dist - d) < 1e-5 and abs(ang - inplane_deg) < 1e-3 and oop < 1e-6
    assert ok
    RESULTS.append({"notebook": "03", "check": f"MonovalentLonePair geometry (inPlane={inplane_deg})",
                    "status": "PASS"})


print("MonovalentLonePair (interchange #939 regression):")
for ip in (100.0, 120.0):
    check_monovalent(ip)

MonovalentLonePair (interchange #939 regression):
  inPlane=100.0: |S-O|=0.400000 (expect 0.4), angle(S,O,C)=100.0000 (expect 100.0), |z|=0.00e+00
  inPlane=100.0: |S-O|=0.400000 (expect 0.4), angle(S,O,C)=100.0000 (expect 100.0), |z|=0.00e+00
  inPlane=120.0: |S-O|=0.400000 (expect 0.4), angle(S,O,C)=120.0000 (expect 120.0), |z|=0.00e+00
  inPlane=120.0: |S-O|=0.400000 (expect 0.4), angle(S,O,C)=120.0000 (expect 120.0), |z|=0.00e+00


### MonovalentLonePair: general out-of-plane placement
The check above pins the in-plane case (`outOfPlaneAngle=0`). Here we drive
**both** angles on an *asymmetric* fragment (so the three parents O, C, C′ are
distinct) and compare to an explicitly-constructed SMIRNOFF frame: origin at O,
in-plane basis from O→C and the plane normal `(O→C) × (O→C′)`, the site rotated
by `inPlaneAngle` in-plane and tilted `outOfPlaneAngle` about the O→C axis. This
exercises the frame math that TIP5P (notebook 02) only covers for symmetric
water.

In [6]:
def check_monovalent_oop(inplane_deg, oop_deg):
    d = 0.45
    coords = np.array([[0, 0, 0], [-1.25, 0, 0], [-1.9, 0.95, 0.0], [-1.9, -1.1, 0.4],
                       [-1.5, -1.9, 0.2], [-2.9, -1.3, 0.1], [-1.7, -1.0, 1.5]])
    vs = (f'<VirtualSite smirks="[#8:1]=[#6:2][#6:3]" type="MonovalentLonePair" name="LP"'
          f' match="all_permutations" distance="{d}*angstrom" inPlaneAngle="{inplane_deg}*degree"'
          f' outOfPlaneAngle="{oop_deg}*degree" charge_increment1="0.0*elementary_charge"'
          f' charge_increment2="0.0*elementary_charge" charge_increment3="0.0*elementary_charge"'
          f' sigma="0*angstrom" epsilon="0*kilocalorie/mole"/>')
    mol, sys, pos = build_ic("[O:1]=[C:2]([H:3])[C:4]([H:5])([H:6])[H:7]", coords, vs, VDW0, LIBQ0)
    O, C, Cp = coords[0], coords[1], coords[3]

    def ref(op_sign):
        ip, op = np.deg2rad(inplane_deg), np.deg2rad(op_sign * oop_deg)
        z1 = (C - O) / np.linalg.norm(C - O)
        znorm = np.cross(z1, Cp - O); znorm /= np.linalg.norm(znorm)
        y = np.cross(znorm, z1); y /= np.linalg.norm(y)
        return O + d * (np.cos(op) * (np.cos(ip) * z1 + np.sin(ip) * y) + np.sin(op) * znorm)

    sites = [pos[i] * 10.0 for i in vsite_indices(sys)]
    dists = [np.linalg.norm(s - O) for s in sites]
    best = min(min(np.linalg.norm(s - ref(sgn)) for s in sites) for sgn in (+1, -1))
    print(f"  inPlane={inplane_deg:5.1f} oop={oop_deg:5.1f}: |S-O|={[f'{x:.4f}' for x in dists]} "
          f"(expect {d}), best match to closed form={best:.2e} A")
    assert all(abs(x - d) < 1e-5 for x in dists) and best < 1e-4
    RESULTS.append({"notebook": "03",
                    "check": f"MonovalentLonePair oop geometry (ip={inplane_deg},oop={oop_deg})",
                    "status": "PASS"})


print("MonovalentLonePair general out-of-plane placement:")
for ip in (95.0, 120.0):
    for oop in (30.0, 50.0):
        check_monovalent_oop(ip, oop)

MonovalentLonePair general out-of-plane placement:
  inPlane= 95.0 oop= 30.0: |S-O|=['0.4500'] (expect 0.45), best match to closed form=5.55e-17 A
  inPlane= 95.0 oop= 50.0: |S-O|=['0.4500'] (expect 0.45), best match to closed form=6.21e-17 A
  inPlane=120.0 oop= 30.0: |S-O|=['0.4500'] (expect 0.45), best match to closed form=6.80e-17 A
  inPlane=120.0 oop= 50.0: |S-O|=['0.4500'] (expect 0.45), best match to closed form=1.14e-16 A


## 2 · Charge-increment bookkeeping

A `BondCharge` site with `charge_increment1=q1`, `charge_increment2=q2`
redistributes charge between the site and its parents. By the SMIRNOFF
convention each parent_i has `q_i` **added** to it and the site carries
`-(q1+q2)`. On top of preset base charges the totals must be exact and net
charge conserved.

In [7]:
def check_charge_bookkeeping():
    base_cl, base_c = -0.05, 0.10   # preset via LibraryCharges
    q1, q2 = 0.20, -0.05
    libq = [("[#17:1]", base_cl), ("[#6:1]", base_c), ("[#1:1]", -(base_cl + base_c) / 3)]
    coords = [[0, 0, 0], [-1.7, 0, 0], [-2.1, 0.9, 0], [-2.1, -0.5, 0.8], [-2.1, -0.5, -0.8]]
    vs = (f'<VirtualSite smirks="[#17:1]-[#6:2]" type="BondCharge" name="EP" match="all_permutations"'
          f' distance="0.5*angstrom" charge_increment1="{q1}*elementary_charge"'
          f' charge_increment2="{q2}*elementary_charge" sigma="0*angstrom" epsilon="0*kilocalorie/mole"/>')
    mol, sys, pos = build_ic("[Cl:1][C:2]([H:3])([H:4])[H:5]", coords, vs, VDW0, libq)
    q = {i: H.particle_nb_params(sys, i)["charge"] for i in range(sys.getNumParticles())}
    cl, c = q[0], q[1]
    site = q[vsite_indices(sys)[0]]
    print(f"  Cl: {cl:+.4f} (expect {base_cl + q1:+.4f})")
    print(f"  C : {c:+.4f} (expect {base_c + q2:+.4f})")
    print(f"  EP: {site:+.4f} (expect {-(q1 + q2):+.4f})")
    print(f"  net charge: {H.total_charge(sys):+.2e} (expect 0)")
    assert abs(cl - (base_cl + q1)) < 1e-6
    assert abs(c - (base_c + q2)) < 1e-6
    assert abs(site - (-(q1 + q2))) < 1e-6
    assert abs(H.total_charge(sys)) < 1e-6
    RESULTS.append({"notebook": "03", "check": "charge-increment bookkeeping", "status": "PASS"})


check_charge_bookkeeping()

  Cl: +0.1500 (expect +0.1500)
  C : +0.0500 (expect +0.0500)
  EP: -0.1500 (expect -0.1500)
  net charge: +3.47e-18 (expect 0)


## 3 · Exclusions and 1-4 scaling (interchange #1436)

We build a four-heavy-atom chain `Cl–C–C–C` with a `BondCharge` site (carrying
both a charge and non-zero Lennard-Jones parameters) off the terminal C–Cl. The
site inherits the parent's exclusions, so it forms a scaled **1-4** pair with
atoms three bonds from its parent. For several `scale14` values we verify the
exact exception parameters: `chargeProd = scale14·q_i·q_j`, the (unscaled)
Lorentz-Berthelot `sigma = (σ_i+σ_j)/2`, and `epsilon = scale14·√(ε_i·ε_j)`. A
linear response in all three is the signature of a correct #1436 fix.

In [8]:
def check_scaling():
    # linear chain Cl-C-C-C with explicit charges AND non-zero Lennard-Jones on
    # every particle (the vsite included), so both the electrostatic and the vdW
    # parts of each scaled vsite 1-4 exception have predictable values.
    smiles = "[Cl:1][C:2]([H:3])([H:4])[C:5]([H:6])([H:7])[C:8]([H:9])([H:10])[H:11]"
    coords = [[0, 0, 0], [-1.7, 0, 0], [-2.1, -0.6, 0.9], [-2.1, -0.6, -0.9],
              [-2.4, 1.3, 0], [-1.9, 2.0, 0.7], [-1.9, 2.0, -0.7],
              [-3.9, 1.3, 0], [-4.3, 0.6, 0.7], [-4.3, 0.6, -0.7], [-4.2, 2.3, 0]]
    q_site1, q_site2 = 0.0, 0.3     # increments -> site carries +0.3
    libq = [("[#17:1]", -0.3), ("[#6:1]", 0.1), ("[#1:1]", 0.0)]
    vdw = [("[#17:1]", 3.4, 0.30), ("[#6:1]", 3.4, 0.10), ("[#1:1]", 2.6, 0.02)]
    vs = (f'<VirtualSite smirks="[#17:1]-[#6:2]" type="BondCharge" name="EP" match="all_permutations"'
          f' distance="0.4*angstrom" charge_increment1="{q_site1}*elementary_charge"'
          f' charge_increment2="{q_site2}*elementary_charge"'
          f' sigma="1.5*angstrom" epsilon="0.05*kilocalorie/mole"/>')

    print("  scale14 : max |chargeProd err| , max |sigma err| , max |epsilon err|  over vsite 1-4 pairs")
    for s14 in (5 / 6, 0.5, 0.25, 1.0):
        ff = ForceField(P.build_offxml(vdw, libq, [vs], elec_scale14=s14, vdw_scale14=s14))
        mol = Molecule.from_mapped_smiles(smiles)
        mol.add_conformer(np.asarray(coords) * ANG)
        sys = ff.create_interchange(mol.to_topology()).to_openmm_system()
        vidx = vsite_indices(sys)[0]
        prm = {i: H.particle_nb_params(sys, i) for i in range(sys.getNumParticles())}
        excs = H.exceptions_table(sys)
        w_q = w_s = w_e = 0.0
        n14 = 0
        for (a, b), e in excs.items():
            if vidx in (a, b) and not e["is_exclusion"]:
                o = a if b == vidx else b
                # electrostatics scaled; sigma is the (unscaled) Lorentz-Berthelot
                # combination; epsilon is scale14 * sqrt(eps_i eps_j)
                exp_q = s14 * prm[vidx]["charge"] * prm[o]["charge"]
                exp_sig = 0.5 * (prm[vidx]["sigma"] + prm[o]["sigma"])
                exp_eps = s14 * np.sqrt(prm[vidx]["epsilon"] * prm[o]["epsilon"])
                w_q = max(w_q, abs(e["chargeProd"] - exp_q))
                w_s = max(w_s, abs(e["sigma"] - exp_sig))
                w_e = max(w_e, abs(e["epsilon"] - exp_eps))
                n14 += 1
        print(f"    {s14:.4f} : {w_q:.2e} , {w_s:.2e} , {w_e:.2e}  ({n14} scaled vsite 1-4 pairs)")
        assert n14 > 0, "expected at least one scaled vsite 1-4 pair"
        assert w_q < 1e-8, f"scale14={s14}: vsite 1-4 chargeProd wrong by {w_q}"
        assert w_s < 1e-6, f"scale14={s14}: vsite 1-4 sigma wrong by {w_s}"
        assert w_e < 1e-8, f"scale14={s14}: vsite 1-4 epsilon wrong by {w_e}"
    RESULTS.append({"notebook": "03", "check": "1-4 scaling on vsite exceptions, charge+LJ (#1436)",
                    "status": "PASS"})


check_scaling()

  scale14 : max |chargeProd err| , max |sigma err| , max |epsilon err|  over vsite 1-4 pairs
    0.8333 : 0.00e+00 , 4.44e-16 , 6.94e-18  (3 scaled vsite 1-4 pairs)
    0.5000 : 0.00e+00 , 4.44e-16 , 6.94e-18  (3 scaled vsite 1-4 pairs)
    0.2500 : 0.00e+00 , 4.44e-16 , 3.47e-18  (3 scaled vsite 1-4 pairs)
    1.0000 : 0.00e+00 , 4.44e-16 , 1.39e-17  (3 scaled vsite 1-4 pairs)


## 3b · Exclusion *set* from first principles

`check_scaling` verifies the *values* of the vsite exceptions that exist; it
does not verify that the right *set* of pairs carries exceptions (a missing 1-4,
or a 1-5 pair wrongly left full-strength, would slip through). Here we derive
the entire exclusion/1-4 pattern from the bare molecular graph — atom–atom by
bond distance, the virtual site cloning orientation-atom-0's relationships and
excluded from its own frame — and require Interchange's exported exceptions to
match exactly. This depends on nothing but the graph, so it is independent of
the export machinery it checks.

In [9]:
# Nonzero Lennard-Jones on every atom type so that a scaled 1-4 pair carries a
# nonzero epsilon and is therefore distinguishable from a true exclusion (both
# would be all-zeros if charges and epsilons were zero). The vsite XML used with
# these checks likewise carries nonzero epsilon.
VDW_NZ = [("[#6:1]", 3.4, 0.10), ("[#1:1]", 2.6, 0.02), ("[#7:1]", 3.3, 0.17),
          ("[#8:1]", 3.0, 0.21), ("[#17:1]", 3.4, 0.30)]


def check_exclusion_set(smiles, coords_A, vsite_xml):
    mol = Molecule.from_mapped_smiles(smiles)
    mol.add_conformer(np.asarray(coords_A) * ANG)
    ff = ForceField(P.build_offxml(VDW_NZ, LIBQ0, [vsite_xml]))
    sys = ff.create_interchange(mol.to_topology()).to_openmm_system()
    bonds = [(b.atom1_index, b.atom2_index) for b in mol.bonds]
    dist = H.graph_distances(mol.n_atoms, bonds)
    # every vsite (there may be several per parent) with its orientation-atom-0
    # and frame; a vsite clones parent0's relationships and is excluded from its
    # own frame and from any co-located vsite sharing a frame atom.
    vinfo = {i: H.vsite_parent0_and_frame(sys, i)
             for i in range(sys.getNumParticles()) if sys.isVirtualSite(i)}

    def rel_atom(a, b):
        d = dist[a][b]
        return "excl" if d <= 2 else ("scaled" if d == 3 else "full")

    def expected(i, j):
        vi, vj = i in vinfo, j in vinfo
        if not vi and not vj:
            return rel_atom(i, j)
        if vi and vj:
            (p_i, _), (p_j, _) = vinfo[i], vinfo[j]
            if p_i == p_j:              # co-parent vsites: zeroed exception
                return "excl"
            return rel_atom(p_i, p_j)   # else clone the parents' relationship
        v, a = (i, j) if vi else (j, i)
        p, frame = vinfo[v]
        return "excl" if a in frame else rel_atom(p, a)

    got = H.exception_kinds(sys)
    diffs = []
    for i in range(sys.getNumParticles()):
        for j in range(i + 1, sys.getNumParticles()):
            exp, act = expected(i, j), got.get((i, j), "full")
            if exp != act:
                diffs.append(((i, j), exp, act))
    print(f"  {smiles}: exceptions held={len(got)}  first-principles disagreements={len(diffs)}")
    for pair, exp, act in diffs[:6]:
        print(f"    pair {pair}: expected {exp}, got {act}")
    assert not diffs, f"exclusion set disagrees with first principles: {diffs[:6]}"
    RESULTS.append({"notebook": "03", "check": f"exclusion set vs first principles ({smiles})",
                    "status": "PASS"})


print("Exclusion set vs first principles (bond-graph derivation):")
_chain = "[Cl:1][C:2]([H:3])([H:4])[C:5]([H:6])([H:7])[C:8]([H:9])([H:10])[H:11]"
_chain_xyz = [[0, 0, 0], [-1.7, 0, 0], [-2.1, -0.6, 0.9], [-2.1, -0.6, -0.9], [-2.4, 1.3, 0],
              [-1.9, 2.0, 0.7], [-1.9, 2.0, -0.7], [-3.9, 1.3, 0], [-4.3, 0.6, 0.7],
              [-4.3, 0.6, -0.7], [-4.2, 2.3, 0]]
_bond_vs = ('<VirtualSite smirks="[#17:1]-[#6:2]" type="BondCharge" name="EP" match="all_permutations"'
            ' distance="0.4*angstrom" charge_increment1="0.0*elementary_charge"'
            ' charge_increment2="0.0*elementary_charge" sigma="1.5*angstrom"'
            ' epsilon="0.05*kilocalorie/mole"/>')
check_exclusion_set(_chain, _chain_xyz, _bond_vs)
# a divalent lone pair (3-parent frame) on dimethyl ether
_ether = "[C:1]([H:4])([H:5])([H:6])[O:2][C:3]([H:7])([H:8])[H:9]"
_ether_xyz = [[1.4, 0, 0], [0, 0, 0], [-0.6, 1.2, 0], [1.8, -0.5, 0.9], [1.8, -0.5, -0.9],
              [2.0, 0.9, 0], [-0.5, 1.8, 0.8], [-0.5, 1.8, -0.8], [-1.6, 1.0, 0]]
_div_vs = ('<VirtualSite smirks="[#8X2:1]([#6:2])[#6:3]" type="DivalentLonePair" name="LP"'
           ' match="all_permutations" distance="0.48*angstrom" outOfPlaneAngle="52*degree"'
           ' charge_increment1="0.0*elementary_charge" charge_increment2="0.0*elementary_charge"'
           ' charge_increment3="0.0*elementary_charge" sigma="1.35*angstrom"'
           ' epsilon="0.05*kilocalorie/mole"/>')
check_exclusion_set(_ether, _ether_xyz, _div_vs)

Exclusion set vs first principles (bond-graph derivation):
  [Cl:1][C:2]([H:3])([H:4])[C:5]([H:6])([H:7])[C:8]([H:9])([H:10])[H:11]: exceptions held=54  first-principles disagreements=0
  [C:1]([H:4])([H:5])([H:6])[O:2][C:3]([H:7])([H:8])[H:9]: exceptions held=46  first-principles disagreements=0


## 3c · Multi-molecule topology — parents stay in-molecule, no cross-molecule leakage

The realistic input is a `Topology` of many (possibly duplicated,
possibly atom-reordered) molecules. A vsite-creation bug using absolute instead
of per-molecule atom indices would show up as a vsite frame spanning two
molecules, a wrong per-molecule exclusion set, or a spurious cross-molecule
exception. We check all three against first principles on a mixed, duplicated,
atom-remapped topology.

In [10]:
def check_multimolecule():
    from openff.toolkit import Topology
    bond = Molecule.from_mapped_smiles(
        "[Cl:1][C:2]([H:6])([H:7])[C:3]([H:8])([H:9])[C:4]([H:10])([H:11])[Cl:5]")
    bond.generate_conformers(n_conformers=1)
    ether = Molecule.from_mapped_smiles(_ether)
    ether.generate_conformers(n_conformers=1)
    plain = Molecule.from_smiles("CCO")
    plain.generate_conformers(n_conformers=1)
    n = bond.n_atoms
    bond_r = bond.remap({i: n - 1 - i for i in range(n)}, current_to_new=True)
    bond_r.generate_conformers(n_conformers=1)
    mols = [bond, ether, plain, bond_r, ether]

    top = Topology()
    for m in mols:
        top.add_molecule(m)
    combo = P.build_offxml(VDW_NZ, LIBQ0, [
        '<VirtualSite smirks="[#6:1]-[#17:2]" type="BondCharge" name="EP" match="all_permutations"'
        ' distance="-0.30*angstrom" charge_increment1="0.0*elementary_charge"'
        ' charge_increment2="0.0*elementary_charge" sigma="1.05*angstrom"'
        ' epsilon="0.03*kilocalorie/mole"/>',
        _div_vs])
    sys = ForceField(combo).create_interchange(top).to_openmm_system()

    atom_mol, gbonds, off = [], [], 0
    for mi, m in enumerate(mols):
        for _ in range(m.n_atoms):
            atom_mol.append(mi)
        for b in m.bonds:
            gbonds.append((off + b.atom1_index, off + b.atom2_index))
        off += m.n_atoms
    dist = H.graph_distances(off, gbonds)

    bad_frame = 0
    p0, frame = {}, {}
    for i in range(sys.getNumParticles()):
        if sys.isVirtualSite(i):
            p0[i], frame[i] = H.vsite_parent0_and_frame(sys, i)
            if len({atom_mol[p] for p in frame[i]}) != 1:
                bad_frame += 1

    def expected(i, j):
        vi, vj = sys.isVirtualSite(i), sys.isVirtualSite(j)
        if vi or vj:
            if vi and vj:
                pi, pj = p0[i], p0[j]
                if atom_mol[pi] != atom_mol[pj]:
                    return "full"
                d = dist[pi][pj]
                return "excl" if d <= 2 else ("scaled" if d == 3 else "full")
            v, a = (i, j) if vi else (j, i)
            if a in frame[v]:
                return "excl"
            if atom_mol[p0[v]] != atom_mol[a]:
                return "full"
            d = dist[p0[v]][a]
            return "excl" if d <= 2 else ("scaled" if d == 3 else "full")
        if atom_mol[i] != atom_mol[j]:
            return "full"
        d = dist[i][j]
        return "excl" if d <= 2 else ("scaled" if d == 3 else "full")

    got = H.exception_kinds(sys)
    diffs = 0
    for i in range(sys.getNumParticles()):
        for j in range(i + 1, sys.getNumParticles()):
            if expected(i, j) != got.get((i, j), "full"):
                diffs += 1
    print(f"  {len(mols)} molecules, {sys.getNumParticles()} particles, {H.count_vsites(sys)} vsites")
    print(f"  frames spanning >1 molecule: {bad_frame}   first-principles disagreements: {diffs}")
    assert bad_frame == 0 and diffs == 0
    assert abs(H.total_charge(sys)) < 1e-6
    RESULTS.append({"notebook": "03", "check": "multi-molecule exclusion set vs first principles",
                    "status": "PASS"})


print("\nMulti-molecule vsite creation (mixed + duplicated + atom-remapped):")
check_multimolecule()


Multi-molecule vsite creation (mixed + duplicated + atom-remapped):


  5 molecules, 57 particles, 8 vsites
  frames spanning >1 molecule: 0   first-principles disagreements: 0


## 4 · Analytic Coulomb energy (nonperiodic)

Two point charges on a chargeless scaffold, with a `BondCharge` site carrying a
known charge, evaluated with `NoCutoff`. We compare the `NonbondedForce` energy
to the hand-summed Coulomb energy over all non-excluded pairs.

In [11]:
def check_analytic_energy():
    # Cl-C with a site on the C (carrying charge), plus a lone probe atom far away
    # that is NOT bonded, so the site<->probe and atom<->probe pairs are
    # full-strength. Two separate molecules => no cross-molecule exclusions.
    # allow_nonintegral_charges lets us use hand-picked charges directly; the
    # analytic Coulomb sum below uses whatever charges the system actually holds.
    from openff.toolkit import Topology
    ke = 332.063713   # kcal*A/(mol*e^2), OpenMM's Coulomb constant
    vdw = VDW0 + [("[#18:1]", 3.4, 0.0)]
    libq = [("[#17:1]", -0.3), ("[#6:1]", 0.1), ("[#1:1]", 0.0), ("[#18:1]", 0.5)]
    vs = ('<VirtualSite smirks="[#17:1]-[#6:2]" type="BondCharge" name="EP" match="all_permutations"'
          ' distance="0.5*angstrom" charge_increment1="0.0*elementary_charge"'
          ' charge_increment2="0.3*elementary_charge" sigma="0*angstrom" epsilon="0*kilocalorie/mole"/>')
    ff = ForceField(P.build_offxml(vdw, libq, [vs]))
    m1 = Molecule.from_mapped_smiles("[Cl:1][C:2]([H:3])([H:4])[H:5]")
    m1.add_conformer(np.array([[0, 0, 0], [-1.7, 0, 0], [-2.1, 0.9, 0],
                               [-2.1, -0.5, 0.8], [-2.1, -0.5, -0.8]]) * ANG)
    probe = Molecule.from_smiles("[Ar]")
    probe.add_conformer(np.array([[8.0, 0, 0]]) * ANG)
    top = Topology.from_molecules([m1, probe])
    sysm = ff.create_interchange(top, allow_nonintegral_charges=True).to_openmm_system()
    coords_A = np.vstack([m1.conformers[0].m_as(offunit.angstrom),
                          probe.conformers[0].m_as(offunit.angstrom)])
    pos = H.resolved_vsite_positions(sysm, coords_A / 10.0 * u.nanometer)
    # gather charges and positions
    n = sysm.getNumParticles()
    q = np.array([H.particle_nb_params(sysm, i)["charge"] for i in range(n)])
    xyz = pos * 10.0  # A
    excl = set()
    for (a, b), e in H.exceptions_table(sysm).items():
        if e["is_exclusion"]:
            excl.add((a, b))
    hand = 0.0
    for i in range(n):
        for j in range(i + 1, n):
            if (i, j) in excl:
                continue
            r = np.linalg.norm(xyz[i] - xyz[j])
            hand += ke * q[i] * q[j] / r
    e_nb, _ = H.compute_energy(sysm, pos, recompute_vsites=True)
    omm = e_nb["components"]["NonbondedForce"].value_in_unit(H.ENERGY_UNIT)
    print(f"  analytic Coulomb = {hand:.6f} kcal/mol   OpenMM NonbondedForce = {omm:.6f}   "
          f"delta = {hand - omm:.2e}")
    assert abs(hand - omm) < 1e-4, "analytic Coulomb energy disagrees with OpenMM"
    RESULTS.append({"notebook": "03", "check": "analytic Coulomb energy", "status": "PASS"})


check_analytic_energy()

  analytic Coulomb = -6.020794 kcal/mol   OpenMM NonbondedForce = -6.020794   delta = 5.42e-09


## 5 · End-to-end nonbonded energy (Coulomb + Lennard-Jones + scaled 1-4)

The check above is Coulomb-only with no 1-4 pairs. Here we close the loop
properly: a `Cl–C–C–C` chain whose `BondCharge` site carries **non-zero LJ and
charge**, giving genuine scaled vsite 1-4 partners, at **non-default** 1-4
scaling. We hand-sum the full nonbonded energy over every pair — Coulomb plus
Lennard-Jones 12-6, with exclusions and 1-4 scaling taken from the bond graph
(not from the system's own exceptions) — and require it to match OpenMM's
`NonbondedForce` energy. This validates the whole placement + charge + exclusion
pipeline against pure hand math.

In [12]:
def check_end_to_end_energy(scale14):
    ke = 332.0637128
    smiles = "[Cl:1][C:2]([H:3])([H:4])[C:5]([H:6])([H:7])[C:8]([H:9])([H:10])[H:11]"
    coords = [[0, 0, 0], [-1.7, 0, 0], [-2.1, -0.6, 0.9], [-2.1, -0.6, -0.9], [-2.4, 1.3, 0],
              [-1.9, 2.0, 0.7], [-1.9, 2.0, -0.7], [-3.9, 1.3, 0], [-4.3, 0.6, 0.7],
              [-4.3, 0.6, -0.7], [-4.2, 2.3, 0]]
    libq = [("[#17:1]", -0.3), ("[#6:1]", 0.1), ("[#1:1]", 0.0)]
    vdw = [("[#17:1]", 3.4, 0.30), ("[#6:1]", 3.4, 0.10), ("[#1:1]", 2.6, 0.02)]
    vs = ('<VirtualSite smirks="[#17:1]-[#6:2]" type="BondCharge" name="EP" match="all_permutations"'
          ' distance="0.4*angstrom" charge_increment1="0.0*elementary_charge"'
          ' charge_increment2="0.3*elementary_charge" sigma="1.5*angstrom" epsilon="0.05*kilocalorie/mole"/>')
    mol = Molecule.from_mapped_smiles(smiles)
    mol.add_conformer(np.asarray(coords) * ANG)
    ff = ForceField(P.build_offxml(vdw, libq, [vs], elec_scale14=scale14, vdw_scale14=scale14))
    sys = ff.create_interchange(mol.to_topology()).to_openmm_system()

    n = sys.getNumParticles()
    prm = [H.particle_nb_params(sys, i) for i in range(n)]
    q = np.array([p["charge"] for p in prm])
    sig = np.array([p["sigma"] for p in prm])
    eps = np.array([p["epsilon"] for p in prm])
    xyz = H.resolved_vsite_positions(sys, mol.conformers[0].to_openmm()) * 10.0  # A

    bonds = [(b.atom1_index, b.atom2_index) for b in mol.bonds]
    dist = H.graph_distances(mol.n_atoms, bonds)
    vidx = vsite_indices(sys)[0]
    parent0, frame = H.vsite_parent0_and_frame(sys, vidx)

    def kind(i, j):
        if vidx in (i, j):
            other = j if i == vidx else i
            if other in frame:
                return "excl"
            d = dist[parent0][other]
        else:
            d = dist[i][j]
        return "excl" if d <= 2 else ("scaled" if d == 3 else "full")

    hand = 0.0
    for i in range(n):
        for j in range(i + 1, n):
            k = kind(i, j)
            if k == "excl":
                continue
            r = np.linalg.norm(xyz[i] - xyz[j])
            s = 0.5 * (sig[i] + sig[j])
            e = np.sqrt(eps[i] * eps[j])
            pair = 4 * e * ((s / r) ** 12 - (s / r) ** 6) + ke * q[i] * q[j] / r
            hand += (scale14 if k == "scaled" else 1.0) * pair

    e_omm, _ = H.compute_energy(sys, mol.conformers[0].to_openmm(), recompute_vsites=True)
    omm = e_omm["components"]["NonbondedForce"].value_in_unit(H.ENERGY_UNIT)
    print(f"  scale14={scale14:.4f}: hand={hand:.6f}  OpenMM={omm:.6f}  delta={hand - omm:+.2e} kcal/mol")
    assert abs(hand - omm) < 1e-4, f"end-to-end energy mismatch {hand - omm}"
    RESULTS.append({"notebook": "03", "check": f"end-to-end nonbonded energy (scale14={scale14:.3f})",
                    "status": "PASS"})


print("End-to-end hand-summed nonbonded energy (Coulomb+LJ, vsite 1-4, non-default scaling):")
for s14 in (5 / 6, 0.6, 0.25, 1.0):
    check_end_to_end_energy(s14)

End-to-end hand-summed nonbonded energy (Coulomb+LJ, vsite 1-4, non-default scaling):

  scale14=0.8333: hand=-3.241367  OpenMM=-3.241367  delta=+5.81e-09 kcal/mol
  scale14=0.6000: hand=-2.358618  OpenMM=-2.358618  delta=+4.19e-09 kcal/mol
  scale14=0.2500: hand=-1.034495  OpenMM=-1.034495  delta=+1.74e-09 kcal/mol
  scale14=1.0000: hand=-3.871902  OpenMM=-3.871902  delta=+6.98e-09 kcal/mol


In [13]:
import json
with open("results_03.json", "w") as fh:
    json.dump(RESULTS, fh, indent=2, default=str)
print(f"\nWrote results_03.json ({len(RESULTS)} rows)")
print("Notebook 03 complete: exported vsite geometry, charge bookkeeping, 1-4 scaling, "
      "and nonbonded energy all match closed-form expectations.")


Wrote results_03.json (21 rows)
Notebook 03 complete: exported vsite geometry, charge bookkeeping, 1-4 scaling, and nonbonded energy all match closed-form expectations.
